# README

- **Author**: RUEI-HSIANG, CHANG（張睿庠；B12303030；AI 協助程式與驗證）
- **Created At**: 2026-11-20
- **Last Modified At**: 2026-11-20

## What does this file do?

僅處理2026公報；81位縣市首長與199位臺北／高雄市議員候選人。核對既有PDF、OCR證據與逐人政見文字，保留未出現在公報者的缺值。既有抽取：[bulletin_2026.ipynb](bulletin_2026.ipynb)。不修改原始PDF或上游CSV。

使用BAAI/bge-m3固定revision，產生全文與標點與條列文字段落兩組1024維、L2正規化向量。以可用時採用 Apple GPU，否則採用 CPU，保存 float32執行。標點段落不是已核實的獨立政見，不計算S5/S6/S7/S10，也不把餘弦相似度當作立場相同的證據。保留topic/subtopic/stance/scope空欄待編碼。

## What does this file take?

- **Source Data Sets**:
  1. `data/processed/candidate_bulletin_2026.csv`；逐人政見與來源；SHA-256 `d2e1c4d5f9d7ef8266ffefd3f5f853eeda35f2174692c03f59b371713adafc7b`。
  2. `data/processed/bulletin_2026_cards_ocr.json`；四格文字層與OCR座標；SHA-256 `4b032748914700f2928e89589481f11261a7b5ca7db500c6d577d1312e8bc563`。
  3. `data/raw/cec_bulletin_2026/file_manifest.json`；所有2026 PDF的精確路徑與hash；SHA-256 `3d1dc56485df2be41b12c53732a2cc21be5464519d6fb02e332ea60d3f9f0aea`。逐檔核對PDF。
- 上游說明與產生程式：`data/code/bulletin_2026.ipynb`。目前CSV與舊PROJECT索引hash不同，原因為CSV列分隔符號LF／CRLF差異，將列分隔符重建為CRLF後可重現舊索引hash，逐人政見去除空白後均與OCR證據一致；不更動其他欄位或重跑Windows OCR。
- 外部模型：[BAAI/bge-m3](https://huggingface.co/BAAI/bge-m3)，revision `5617a9f61b028005a4858fdac845db406aefb181`；使用官方tokenizer與CLS pooling設定。權重放快取，禁止提交大型權重。

## What does this file output?

以下均位於`data/processed/`，前綴`bulletin_embeddings_2026_`；role為探索用語料／向量，不是已完成的策略變數。

- `corpus.csv`：280位候選人、原文、來源、token數與向量列號；缺文者無向量。
- `segments.csv`：標點段落、原文起訖字元位置、候選人歸屬與待編碼欄；段落ID唯一。
- `candidate_vectors.npy`、`segment_vectors.npy`：float32向量；列號分別對應corpus/segments中的embedding_row。
- `neighbors.csv`：每段文字在其他候選人的最近鄰；純探索，不設分類門檻。
- `diagnostics.csv`：人工構造之同議題／反立場等診斷句；不來自公報，不納入候選人資料。
- `run.json`：模型revision、套件版本、來源／輸出hash與實際檢查結果。
- Documentation: `data/processed/bulletin_embeddings_2026_documentation.ipynb`；hash及重新讀檔驗證由該文件逐檔呈現。

All paths are relative to the project root. rerun會重建本組明列輸出；既有來源不覆寫。完整公報政策文字包含口號、圖中人物名字、OCR錯字與多欄交錯風險；目前未逐人逐字人工審核。


### 圖片政見增補

針對184張圖片式政見，以5倍解析度重新裁切政見欄，使用Tesseract 的chi_tra 繁體中文辨識；保存每行位置與信心值。程式為 [bulletin_ocr_2026.py](bulletin_ocr_2026.py)。固定輸入 `data/processed/bulletin_embeddings_2026_ocr.json` 的 SHA-256 為 `b1ee816896056cbd56729a48ab1d0eff671ef97e12014fe97c9e698f272891af`。使用 Tesseract 5.5.3 與 tessdata_fast chi_tra（訓練檔 SHA-256 固定在程式內）。第二種讀法至少有8個中文字且達舊讀法1.5倍時採用，兩者中文字序列比對低於0.5時標記優先回查，全文及舊版原文皆保留。完整方法與逐筆限制見對應說明。


In [1]:
from pathlib import Path
import sys, json
for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    if (candidate / 'AGENTS.md').is_file() and (candidate / 'templates').is_dir():
        PROJECT_ROOT = candidate
        break
else:
    raise FileNotFoundError('Open within project')
sys.path.insert(0, str(PROJECT_ROOT / 'data/code'))
import bulletin_embeddings_2026 as pipeline


## Processing

使用配對輔助程式的實際流程；輸入hash、2026年、PDF hash、OCR原文一致、token上限及向量正規化均須通過。模型以固定revision下載，或指定`DSSI_BGE_MODEL_DIR`至同revision本機快取。

In [2]:
config = pipeline.run('5617a9f61b028005a4858fdac845db406aefb181')

/tmp/dssi-bulletin-embeddings-venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


W1120 14:54:12.367000 79755 /private/tmp/dssi-bulletin-embeddings-venv/lib/python3.12/site-packages/torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 88303.51it/s]

Embedding candidate policy sections: 279


/Volumes/Crucial X9/DS SI/course project/data-science-course/data/code/bulletin_embeddings_2026.py:261: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  assert model.get_sentence_embedding_dimension() == 1024


Embedding punctuation segments: 3598


{
  "model_id": "BAAI/bge-m3",
  "revision": "5617a9f61b028005a4858fdac845db406aefb181",
  "dimensions": 1024,
  "max_sequence_tokens": 8192,
  "batch_size": 8,
  "device": "mps",
  "dtype": "float32",
  "normalize_embeddings": true,
  "similarity": "cosine",
  "prompt": null,
  "segmentation": "bullet and punctuation v2: 。！？；!?; boundaries; layout whitespace collapsed; commas/newlines not split",
  "scope": "2026 only; 81 chiefs + 199 Taipei/Kaohsiung council candidates",
  "input_hashes": {
    "data/processed/bulletin_embeddings_2026_ocr.json": "b1ee816896056cbd56729a48ab1d0eff671ef97e12014fe97c9e698f272891af",
    "data/processed/candidate_bulletin_2026.csv": "d2e1c4d5f9d7ef8266ffefd3f5f853eeda35f2174692c03f59b371713adafc7b",
    "data/processed/bulletin_2026_cards_ocr.json": "4b032748914700f2928e89589481f11261a7b5ca7db500c6d577d1312e8bc563",
    "data/raw/cec_bulletin_2026/file_manifest.json": "3d1dc56485df2be41b12c53732a2cc21be5464519d6fb02e332ea60d3f9f0aea"
  },
  "manifest_pdf_

## Saved-output verification

In [3]:
import numpy as np, pandas as pd
for rel, expected in config['output_hashes'].items():
    assert pipeline.sha(PROJECT_ROOT / rel) == expected
print('All saved output hashes verified')
pd.read_csv(PROJECT_ROOT / 'data/processed/bulletin_embeddings_2026_diagnostics.csv')

All saved output hashes verified


,pair_type,text_a,text_b,cosine_similarity,source
0,同議題同立場,支持重啟核電廠。,贊成恢復核能發電。,0.920354,synthetic_diagnostic_not_bulletin
1,同議題反立場,支持重啟核電廠。,反對重啟核電廠。,0.847016,synthetic_diagnostic_not_bulletin
2,不同議題,支持重啟核電廠。,增設公共托育中心。,0.582849,synthetic_diagnostic_not_bulletin
3,同議題不同措施,興建社會住宅。,擴大租金補貼。,0.675176,synthetic_diagnostic_not_bulletin
4,同議題不同尺度,改善本市公共運輸。,改革全國公共運輸補助制度。,0.793210,synthetic_diagnostic_not_bulletin
